# 01 — Trajectory Analysis

Behavioral clustering and metric analysis of RL agent trajectories.

**Prerequisites:** Run `python interpretability/run_collection.py --model-path artifacts/good_old/ckpt_best.pt` first.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats
import umap
import hdbscan
import torch

from cogniland.env.types import EnvConfig
from interpretability.data_manager import TrajectoryDataManager
from interpretability.trajectory_features import featurize_all
from interpretability import viz

%matplotlib inline
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14, 'figure.dpi': 120})
sns.set_style('whitegrid')

In [ ]:
# Load data
dm = TrajectoryDataManager('../../interpretability/data/')
print(f'{dm.n_trajectories} trajectories loaded')
dm.summary.head()

In [ ]:
# Compile terrain data for colors
env_config = EnvConfig(device='cpu')
compiled = env_config.compile_terrain('cpu')

## 1. Featurize Trajectories

In [ ]:
features, feature_names = featurize_all(
    dm.summary, str(dm.h5_path), max_episode_length=1000
)
print(f'Feature matrix: {features.shape}')
print(f'Features: {feature_names}')

## 2. PCA + UMAP

In [ ]:
# PCA first (keep 95% variance)
pca = PCA()
pca.fit(features)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_keep = int(np.searchsorted(cumvar, 0.95)) + 1
n_keep = max(n_keep, 2)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cumvar, 'o-')
ax.axhline(0.95, color='r', linestyle='--', label='95%')
ax.axvline(n_keep - 1, color='g', linestyle='--', label=f'n={n_keep}')
ax.set_xlabel('Component')
ax.set_ylabel('Cumulative Variance')
ax.set_title('PCA Explained Variance')
ax.legend()
plt.show()

features_pca = pca.transform(features)[:, :n_keep]
print(f'Keeping {n_keep} PCA components')

In [ ]:
# UMAP
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42)
embedding_umap = reducer.fit_transform(features_pca)

# Also t-SNE for comparison
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=min(30, len(features_pca) - 1), random_state=42)
embedding_tsne = tsne.fit_transform(features_pca)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
ax1.scatter(embedding_umap[:, 0], embedding_umap[:, 1], s=20, alpha=0.7)
ax1.set_title('UMAP')
ax2.scatter(embedding_tsne[:, 0], embedding_tsne[:, 1], s=20, alpha=0.7)
ax2.set_title('t-SNE')
plt.tight_layout()
plt.show()

## 3. HDBSCAN Clustering

In [ ]:
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=max(3, len(features) // 10),
    min_samples=2,
)
labels = clusterer.fit_predict(embedding_umap)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
print(f'Found {n_clusters} clusters ({(labels == -1).sum()} noise points)')

# Add to summary
dm.summary['cluster'] = labels

In [ ]:
# UMAP colored by cluster
fig = viz.plot_umap_embedding(
    embedding_umap, labels=labels,
    title='Strategy Clusters (UMAP + HDBSCAN)'
)
plt.show()

In [ ]:
# UMAP colored by behavioral metrics
metrics_to_show = ['directness_ratio', 'ocean_usage_ratio', 'forest_usage_ratio',
                   'risk_score', 'episode_length', 'average_hp']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for i, col in enumerate(metrics_to_show):
    ax = axes[i // 3][i % 3]
    vals = dm.summary[col].values if col in dm.summary.columns else np.zeros(len(embedding_umap))
    viz.plot_umap_embedding(embedding_umap, continuous_color=vals, title=col, ax=ax)
fig.tight_layout()
plt.show()

## 4. Cluster Interpretation

In [ ]:
# Per-cluster metric means
metric_cols = ['directness_ratio', 'risk_score', 'ocean_usage_ratio',
               'forest_usage_ratio', 'episode_length', 'average_hp',
               'average_resources', 'map_coverage', 'total_return']
metric_cols = [c for c in metric_cols if c in dm.summary.columns]

cluster_means = dm.summary.groupby('cluster')[metric_cols].mean()
print('Cluster-wise metric means:')
display(cluster_means.round(3))

In [ ]:
# Representative trajectories per cluster
test_maps_path = '../../data/test_seed42_n16.pt'
beh_maps_path = '../../data/test_behavior.pt'

unique_labels = sorted(set(labels[labels >= 0]))
for lab in unique_labels:
    cluster_df = dm.summary[dm.summary['cluster'] == lab]
    print(f'\nCluster {lab}: {len(cluster_df)} trajectories')
    
    # Show 2 example trajectories
    for _, row in cluster_df.head(2).iterrows():
        tid = int(row['traj_id'])
        try:
            wmap, spawn, target = dm.get_map_for_trajectory(tid, test_maps_path, beh_maps_path)
            tdata = dm.get_trajectory(tid)
            fig = viz.plot_trajectory_on_map(
                wmap, tdata['positions'], compiled,
                spawn=spawn, target=target,
                title=f'C{lab} T{tid} ({row["outcome"]})',
                figsize=(6, 6),
            )
            plt.show()
        except Exception as e:
            print(f'  Could not render T{tid}: {e}')

## 5. Statistical Analysis

In [ ]:
# Correlation matrix (Spearman)
corr_cols = [c for c in ['directness_ratio', 'risk_score', 'ocean_usage_ratio',
                          'forest_usage_ratio', 'episode_length', 'average_hp',
                          'average_resources', 'map_coverage', 'total_return',
                          'min_hp'] if c in dm.summary.columns]

fig = viz.plot_correlation_matrix(dm.summary, columns=corr_cols)
plt.show()

In [ ]:
# Successful vs failed: Mann-Whitney U tests
success = dm.summary[dm.summary['outcome'] == 'success']
failed = dm.summary[dm.summary['outcome'] != 'success']

print(f'Successful: {len(success)}, Failed: {len(failed)}')
print(f'\n{"Metric":<25} {"U-stat":>10} {"p-value":>12} {"Effect":>10}')
print('-' * 60)

for col in metric_cols:
    s_vals = success[col].dropna()
    f_vals = failed[col].dropna()
    if len(s_vals) > 1 and len(f_vals) > 1:
        u_stat, p_val = stats.mannwhitneyu(s_vals, f_vals, alternative='two-sided')
        direction = 'S>F' if s_vals.mean() > f_vals.mean() else 'F>S'
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else ''
        print(f'{col:<25} {u_stat:>10.0f} {p_val:>12.4g} {direction:>6} {sig}')

In [ ]:
# Scatter plots of key pairs
pairs = [
    ('ocean_usage_ratio', 'episode_length'),
    ('risk_score', 'total_return'),
    ('forest_usage_ratio', 'average_hp'),
    ('directness_ratio', 'total_return'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for idx, (x_col, y_col) in enumerate(pairs):
    ax = axes[idx // 2][idx % 2]
    if x_col in dm.summary.columns and y_col in dm.summary.columns:
        for lab in sorted(dm.summary['cluster'].unique()):
            mask = dm.summary['cluster'] == lab
            label = f'C{lab}' if lab >= 0 else 'Noise'
            ax.scatter(dm.summary.loc[mask, x_col], dm.summary.loc[mask, y_col],
                      s=30, alpha=0.7, label=label)
        ax.set_xlabel(x_col)
        ax.set_ylabel(y_col)
        ax.legend(fontsize=8)
fig.tight_layout()
plt.show()